In [1]:

import plotly.graph_objs as go
import pandas as pd
from collections import Counter
import re
import unicodedata

In [ ]:
import pandas as pd
import re
import unicodedata
import nltk
from nltk.corpus import stopwords

# Baixa as stopwords do NLTK (descomente na primeira execução, se necessário)
nltk.download('stopwords')

# ================= CONFIG E LOAD =================
ARQUIVO_PARQUET = 'dados/cnpqBolsasAuxilios.parquet'
df = pd.read_parquet(ARQUIVO_PARQUET)



# ================= STOPWORDS =================
STOPWORDS_EXTRA = {
    # === genéricas acadêmicas ===
    'projeto', 'projetos', 'pesquisa', 'pesquisas', 'pesquisador', 'pesquisadores',
    'estudo', 'analise', 'avaliacao', 'proposta', 'objetivo', 'objetivos',
    'metodologia', 'resultado', 'resultados',
    # === estrutura institucional ===
    'cnpq', 'programa', 'programas', 'edital', 'editais', 'instituicao', 'instituicoes',
    'universidade', 'universidades', 'departamento', 'centro', 'curso', 'instituto', 'federal', 'nacional',
    # === tempo / contexto ===
    'ano', 'anos', 'mes', 'meses', 'periodo', 'inicio', 'fim', 'duracao',
    # === financiamento ===
    'bolsa', 'bolsas', 'bolsista', 'bolsistas', 'apoio', 'auxilio', 'financiamento',
    'fomento', 'recursos', 'valor', 'valores',
    # === estrutura textual ===
    'atividade', 'atividades', 'relatorio', 'informacao',
    # === ensino genérico ===
    'ensino', 'formacao', 'graduacao',
    # === geográfico ===
    'brasil', 'estado',
    # === números comuns ===
    '2022', '2023', '2024'
}

# 1. Pega as stopwords padrão em português (para, a, de, com, etc)
stopwords_nltk = stopwords.words('portuguese')

# 2. IMPORTANTE: Remove os acentos das stopwords do NLTK!
# Como a sua função "aplicar_stopwords" remove os acentos do texto antes de filtrar,
# as stopwords também precisam estar sem acento para o "match" funcionar (ex: "são" vira "sao").
stopwords_nltk_limpas = {
    unicodedata.normalize('NFKD', w).encode('ASCII', 'ignore').decode('utf-8').lower()
    for w in stopwords_nltk
}

# 3. Une as duas listas
STOPWORDS = set(STOPWORDS_EXTRA).union(stopwords_nltk_limpas)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\felip\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [11]:
# aplicando stopwords no dataframe
def aplicar_stopwords(texto):
    if pd.isna(texto):
        return texto
    
    # normalizar texto: remover acentos, converter para minúsculas
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')
    texto = texto.lower()
      
    # remover stopwords
    palavras = re.findall(r'\b\w+\b', texto)
    palavras_filtradas = [palavra for palavra in palavras if palavra not in STOPWORDS]
    
    return ' '.join(palavras_filtradas)


df = df.copy()
df['palavras_chave_limpo'] = df['palavra_chave'].apply(aplicar_stopwords)
df['titulo_limpo'] = df['titulo_do_projeto'].apply(aplicar_stopwords)

df.drop(columns=['palavra_chave', 'titulo_do_projeto'], inplace=True)
df.rename(columns={
    'palavras_chave_limpo': 'palavra_chave',
    'titulo_limpo': 'titulo_do_projeto'
}, inplace=True)

df.to_parquet('dados/cnpqBolsasAuxilios_limpo.parquet', index=False)

df.head()


,_record_number,ano_referencia,linha_de_fomento,modalidade,categoria_nivel,programa_cnpq,grande_area,area,subarea,instituicao_origem,...,regiao_destino,pais_destino,uo,natureza_de_despesa,valor_pago,cod_modalidade,categoria,tipo_modalidade,palavra_chave,titulo_do_projeto
0,3167657,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,C,Programa Institutos Nacionais de Ciência e Tec...,Ciências Exatas e da Terra,Química,Físico-Química,Universidade Federal de Minas Gerais,...,SE,BRA - Brasil,NaN,NaN,13750.0,NaN,Tecnologia - Desenvolvimento,Desenvolvimento Tecnológico,materiais renovaveis tecnologias ambientais re...,inct midas tecnologias ambientais valoracao re...
1,3167658,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,A,Programa Especial de Cooperação com o Ministér...,Ciências da Saúde,Saúde Coletiva,Saúde Pública,SENAI - Departamento Regional da Bahia,...,NE,BRA - Brasil,NaN,NaN,2400.0,NaN,Tecnologia - Desenvolvimento,Desenvolvimento Tecnológico,covid 19 lion clinico nanocarreador lipidico t...,estudos clinicos fase i ii eficacia seguranca ...
2,3167659,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,B,Programa Institutos Nacionais de Ciência e Tec...,Engenharias,Engenharia Biomédica,Bioengenharia,Universidade de Sao Paulo,...,SE,BRA - Brasil,NaN,NaN,19500.0,NaN,Tecnologia - Desenvolvimento,Desenvolvimento Tecnológico,processamento imagens medicas modelagem simula...,ciencia tecnologia medicina assistida computac...
3,3167660,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,C,Programa Institutos Nacionais de Ciência e Tec...,Ciências Exatas e da Terra,Química,Físico-Química,Universidade Federal de Minas Gerais,...,SE,BRA - Brasil,NaN,NaN,3300.0,NaN,Tecnologia - Desenvolvimento,Desenvolvimento Tecnológico,materiais renovaveis tecnologias ambientais re...,inct midas tecnologias ambientais valoracao re...
4,3167661,2023.0,APOIO A PROJETOS DE PESQUISA,DTI - Desenvolvimento Tecnológico Industrial,B,PROGRAMA DE AÇÕES VINCULADAS À BIOTECNOLOGIA E...,Ciências da Saúde,Medicina,Clínica Médica,Universidade de Sao Paulo,...,SE,BRA - Brasil,NaN,NaN,7800.0,NaN,Tecnologia - Desenvolvimento,Desenvolvimento Tecnológico,regeneracao cardiaca terapia celular celula tr...,cardiomiocitos derivados ipsc humanas hipsc cm...
